# GB Power Imbalance Risk Agent

## Step 1: prove the data works

The eventual model will estimate the probability that the GB electricity system is short in each settlement period tomorrow. 

This notebook does not build the model yet. First, it downloads seven complete days and checks that every forecast was published before our prediction cutoff. If this part is wrong, everything after it is wrong.

## 1. Imports

I only need standard date tools, pandas and requests. `ZoneInfo` makes the cutoff follow London daylight-saving time automatically.

In [12]:
from datetime import date, datetime, time, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import requests

## 2. Settings

I use 16:00 London time on the previous day as the prediction cutoff. The seven-day sample is in January, so every sample day has the normal 48 settlement periods. 

I will test the daylight-saving exceptions separately when I expand the data.

In [13]:
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1"
LONDON = ZoneInfo("Europe/London")
CUTOFF_TIME = time(16, 0)

SAMPLE_START = date(2025, 1, 2)
NUMBER_OF_DAYS = 7

HEADERS = {
    "User-Agent": "gb-poIr-risk-agent/0.1 educational-project"
}

session = requests.Session()
session.headers.update(HEADERS)

## 3. Work out the cutoff

For a target date such as 2 January, the model may only use information published by 16:00 London time on 1 January. I convert that timestamp to UTC because the Elexon API uses UTC.

In [14]:
def prediction_cutoff(target_date):
    previous_day = target_date - timedelta(days=1)
    local_cutoff = datetime.combine(previous_day, CUTOFF_TIME, LONDON)
    return pd.Timestamp(local_cutoff).tz_convert("UTC")


prediction_cutoff(SAMPLE_START)

Timestamp('2025-01-01 16:00:00+0000', tz='UTC')

## 4. Download one published forecast

Elexon can return several versions of the same forecast. This function keeps the latest version that was published before the cutoff.

I keep the publication timestamp beside the value to keep the leakage check easily visible.

In [15]:
def fetch_forecast(dataset, target_date, value_column, boundary=None):
    cutoff = prediction_cutoff(target_date)

    params = {
        "publishDateTimeFrom": cutoff.normalize().isoformat().replace("+00:00", "Z"),
        "publishDateTimeTo": cutoff.isoformat().replace("+00:00", "Z"),
        "format": "json",
    }

    if boundary is not None:
        params["boundary"] = boundary

    response = session.get(
        f"{BASE_URL}/datasets/{dataset}",
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    data = pd.DataFrame(response.json()["data"])

    if data.empty:
        raise ValueError(f"No {dataset} data returned for {target_date}")

    data["publishTime"] = pd.to_datetime(data["publishTime"], utc=True)
    data["startTime"] = pd.to_datetime(data["startTime"], utc=True)

    if "settlementDate" in data.columns:
        data = data[data["settlementDate"] == target_date.isoformat()]
    else:
        data = data[data["startTime"].dt.date == target_date]

    data = data[data["publishTime"] <= cutoff]
    data = data.sort_values("publishTime")
    data = data.drop_duplicates("startTime", keep="last")

    publication_column = f"{dataset.lower()}_published_at"

    return data[["startTime", "publishTime", value_column]].rename(
        columns={"publishTime": publication_column}
    )

## 5. Download the realised outcome

Net Imbalance Volume is the target, not a feature. Positive Net Imbalance Volume means the system was short.

The outcome is published after the settlement period. That is allowed because it is used only as the answer the model will later try to predict.

In [16]:
def fetch_outcome(target_date):
    response = session.get(
        f"{BASE_URL}/balancing/settlement/system-prices/{target_date.isoformat()}",
        timeout=60,
    )
    response.raise_for_status()

    outcome = pd.DataFrame(response.json()["data"])
    outcome["startTime"] = pd.to_datetime(outcome["startTime"], utc=True)
    outcome["createdDateTime"] = pd.to_datetime(
        outcome["createdDateTime"], utc=True
    )

    return outcome[[
        "startTime",
        "settlementPeriod",
        "netImbalanceVolume",
        "systemBuyPrice",
        "createdDateTime",
    ]]

## 6. Build one prediction day

For each day I download four forecasts:

- NDF: national demand forecast
- WINDFOR: wind generation forecast
- IMBALNGC: indicated system imbalance
- MELNGC: indicated system margin

The wind forecast is hourly. I carry each hourly value forward to the matching half-hour period. The other forecasts already use settlement-period resolution.

In [17]:
def build_one_day(target_date):
    demand = fetch_forecast("NDF", target_date, "demand", boundary="N")
    wind = fetch_forecast("WINDFOR", target_date, "generation")
    imbalance = fetch_forecast(
        "IMBALNGC", target_date, "imbalance", boundary="N"
    )
    margin = fetch_forecast("MELNGC", target_date, "margin", boundary="N")

    day = pd.merge_asof(
        demand.sort_values("startTime"),
        wind.sort_values("startTime"),
        on="startTime",
        direction="backward",
        tolerance=pd.Timedelta("30min"),
    )

    day = day.merge(imbalance, on="startTime", how="left")
    day = day.merge(margin, on="startTime", how="left")
    day = day.merge(fetch_outcome(target_date), on="startTime", how="left")

    day["system_short"] = (day["netImbalanceVolume"] > 0).astype("Int64")
    day["settlement_date"] = target_date.isoformat()
    day["prediction_cutoff"] = prediction_cutoff(target_date)

    return day

## 7. Download seven days

This makes 35 API calls in total. It can take around one minute. The printed date shows which day is being downloaded.

In [18]:
sample_days = [
    SAMPLE_START + timedelta(days=offset)
    for offset in range(NUMBER_OF_DAYS)
]

daily_data = []

for target_date in sample_days:
    print(f"Downloading {target_date}")
    daily_data.append(build_one_day(target_date))

sample = pd.concat(daily_data, ignore_index=True)
sample.shape

(336, 16)

## 8. Run the leakage and quality checks

The checks are intentionally simple. I want to see the exact reason the sample passes or fails.

In [19]:
publication_columns = [
    "ndf_published_at",
    "windfor_published_at",
    "imbalngc_published_at",
    "melngc_published_at",
]

critical_columns = [
    "demand",
    "generation",
    "imbalance",
    "margin",
    "netImbalanceVolume",
]

rows_per_day = sample.groupby("settlement_date").size()
duplicate_keys = sample.duplicated(
    ["settlement_date", "settlementPeriod"]
).sum()
missing_values = int(sample[critical_columns].isna().sum().sum())
late_forecasts = {
    column: int((sample[column] > sample["prediction_cutoff"]).sum())
    for column in publication_columns
}

audit = pd.Series({
    "rows": len(sample),
    "days": sample["settlement_date"].nunique(),
    "minimum periods in one day": int(rows_per_day.min()),
    "maximum periods in one day": int(rows_per_day.max()),
    "duplicate keys": int(duplicate_keys),
    "missing critical values": missing_values,
    "forecasts published after cutoff": sum(late_forecasts.values()),
    "share of periods with short system": sample["system_short"].mean(),
})

audit

rows                                  336.000000
days                                    7.000000
minimum periods in one day             48.000000
maximum periods in one day             48.000000
duplicate keys                          0.000000
missing critical values                 0.000000
forecasts published after cutoff        0.000000
share of periods with short system      0.595238
dtype: float64

## 9. Stop if a critical check fails

For this January sample I expect exactly 48 settlement periods per day. Later, the full-history code will explicitly allow the 46-period and 50-period daylight-saving days.

In [20]:
assert len(sample) == 7 * 48, "Unexpected number of settlement periods"
assert duplicate_keys == 0, "Duplicate date and settlement-period keys found"
assert missing_values == 0, "Critical values are missing"
assert sum(late_forecasts.values()) == 0, "A forecast was published after the cutoff"

print("PASS: the seven-day sample is complete and point-in-time safe.")

PASS: the seven-day sample is complete and point-in-time safe.


## 10. Inspect and save the sample

The publication columns are kept in the saved file. We will use them again when testing the full historical dataset.

In [21]:
columns_to_show = [
    "settlement_date",
    "settlementPeriod",
    "startTime",
    "demand",
    "generation",
    "imbalance",
    "margin",
    "netImbalanceVolume",
    "system_short",
]

sample[columns_to_show].head(10)

,settlement_date,settlementPeriod,startTime,demand,generation,imbalance,margin,netImbalanceVolume,system_short
0,2025-01-02,1,2025-01-02 00:00:00+00:00,23200,9422,912,42225,453.248281,1
1,2025-01-02,2,2025-01-02 00:30:00+00:00,23500,9422,798,42079,508.067025,1
2,2025-01-02,3,2025-01-02 01:00:00+00:00,23197,9218,746,42125,521.590516,1
3,2025-01-02,4,2025-01-02 01:30:00+00:00,22726,9218,1186,42618,75.652149,1
4,2025-01-02,5,2025-01-02 02:00:00+00:00,22306,8916,1439,42972,-142.770411,0
5,2025-01-02,6,2025-01-02 02:30:00+00:00,21791,8916,1800,43495,337.665234,1
6,2025-01-02,7,2025-01-02 03:00:00+00:00,21589,8899,1961,43664,389.852190,1
7,2025-01-02,8,2025-01-02 03:30:00+00:00,21250,8899,2359,44048,-140.030130,0
8,2025-01-02,9,2025-01-02 04:00:00+00:00,21071,8938,2516,44012,-49.170263,0
9,2025-01-02,10,2025-01-02 04:30:00+00:00,21000,8938,3013,44114,-21.574998,0


In [22]:
sample.to_csv("seven_day_sample.csv", index=False)
print("Saved seven_day_sample.csv")

Saved seven_day_sample.csv


For these seven days, I can reconstruct the demand, wind, indicated imbalance and margin forecasts that were available by 16:00 London time on the previous day. We can then match them to the realised Net Imbalance Volume without using the outcome as a feature.

The next step is to expand the history and handle the two daylight-saving transition days correctly. I still do not need the ML model or Ollama agent yet.